# Notebook 01: Alkene VQE Simulation
## Quantum Simulation of Alkenes via PySCF + OpenFermion + PennyLane

**Molecules:** Ethylene (C₂H₄), 1-Butene (C₄H₈)
**Methods:** Hartree-Fock → CCSD → VQE (UCCSD ansatz)
**Mapping:** Jordan-Wigner and Bravyi-Kitaev

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/quantum-alkene-alkyne-pyscf/blob/main/notebooks/01_alkene_vqe_simulation.ipynb)

In [ ]:
# ── CELL 1: Install dependencies (run once) ──────────────────────────
import sys
!pip install -q pyscf openfermion openfermionpyscf pennylane pennylane-qiskit qiskit qiskit-aer 2>&1 | tail -5
print('Dependencies installed.')

In [ ]:
# ── CELL 2: Imports ───────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pyscf import gto, scf, ci, cc
from openfermion.chem import MolecularData
from openfermion import get_fermion_operator, jordan_wigner, bravyi_kitaev
from openfermion.utils import count_qubits
from openfermionpyscf import run_pyscf
import pennylane as qml
from pennylane import qchem
print('All imports successful.')

In [ ]:
# ── CELL 3: Define Ethylene geometry ─────────────────────────────────
ethylene_geometry = [
    ('C', (0.000,  0.000,  0.000)),
    ('C', (0.000,  0.000,  1.339)),
    ('H', (0.000,  0.926, -0.546)),
    ('H', (0.000, -0.926, -0.546)),
    ('H', (0.000,  0.926,  1.885)),
    ('H', (0.000, -0.926,  1.885)),
]

BASIS = 'sto-3g'  # minimal basis; change to '3-21g' or '6-31g' for better accuracy

ethylene_data = MolecularData(
    geometry=ethylene_geometry,
    basis=BASIS,
    multiplicity=1,
    charge=0,
    description='ethylene',
)
print(f'Ethylene: {ethylene_data.n_atoms} atoms, basis={BASIS}')

In [ ]:
# ── CELL 4: Run classical HF, CCSD, FCI calculations ─────────────────
ethylene_data = run_pyscf(ethylene_data, run_scf=True, run_ccsd=True, run_fci=True)

print(f"Ethylene STO-3G Classical Energies:")
print(f"  HF   energy : {ethylene_data.hf_energy:.8f} Ha")
print(f"  CCSD energy : {ethylene_data.ccsd_energy:.8f} Ha")
print(f"  FCI  energy : {ethylene_data.fci_energy:.8f} Ha")
print(f"  n_electrons : {ethylene_data.n_electrons}")
print(f"  n_orbitals  : {ethylene_data.n_orbitals}")
print(f"  n_qubits (full JW): {ethylene_data.n_qubits}")

In [ ]:
# ── CELL 5: Generate qubit Hamiltonian (Jordan-Wigner) ────────────────
fermion_ham = get_fermion_operator(ethylene_data.get_molecular_hamiltonian())
jw_ham = jordan_wigner(fermion_ham)
n_qubits_jw = count_qubits(jw_ham)

bk_ham = bravyi_kitaev(fermion_ham)
n_qubits_bk = count_qubits(bk_ham)

print(f'JW mapping: {n_qubits_jw} qubits')
print(f'BK mapping: {n_qubits_bk} qubits')
print(f'Number of Pauli terms (JW): {len(list(jw_ham.terms))}')

In [ ]:
# ── CELL 6: Active space selection ──────────────────────────────────
# For hardware feasibility, select active space around HOMO/LUMO
# Ethylene: 16 electrons, 7 orbitals (STO-3G) → reduce to 4e/4o active space

from openfermion.transforms import freeze_orbitals

# Freeze 6 core electrons (3 doubly-occupied orbitals)
active_fermion_ham = freeze_orbitals(fermion_ham, occupied=[0,1,2], virtual=[])
active_jw_ham = jordan_wigner(active_fermion_ham)
n_qubits_active = count_qubits(active_jw_ham)

print(f'Full space: {n_qubits_jw} qubits')
print(f'Active space (3 frozen core): {n_qubits_active} qubits')
print('Active space is more feasible for current NISQ hardware.')

In [ ]:
# ── CELL 7: Build PennyLane Hamiltonian ───────────────────────────────
# Convert OpenFermion QubitOperator → PennyLane Hamiltonian

def openfermion_to_pennylane(qubit_op, n_qubits):
    """Convert OpenFermion QubitOperator to PennyLane Hamiltonian."""
    coeffs = []
    ops = []
    for term, coeff in qubit_op.terms.items():
        coeffs.append(np.real(coeff))
        if len(term) == 0:
            ops.append(qml.Identity(0))
        else:
            pauli_list = []
            for qubit_idx, pauli in term:
                gate = {'X': qml.PauliX, 'Y': qml.PauliY, 'Z': qml.PauliZ}[pauli]
                pauli_list.append(gate(qubit_idx))
            if len(pauli_list) == 1:
                ops.append(pauli_list[0])
            else:
                ops.append(qml.operation.Tensor(*pauli_list))
    return qml.Hamiltonian(coeffs, ops)

H_pl = openfermion_to_pennylane(active_jw_ham, n_qubits_active)
print(f'PennyLane Hamiltonian has {len(H_pl.coeffs)} terms.')

In [ ]:
# ── CELL 8: VQE with PennyLane (statevector simulator) ───────────────
n_electrons_active = ethylene_data.n_electrons - 6  # subtract frozen
singles, doubles = qchem.excitations(n_electrons_active, n_qubits_active)
hf_state = qchem.hf_state(n_electrons_active, n_qubits_active)

dev = qml.device('default.qubit', wires=n_qubits_active)

@qml.qnode(dev)
def vqe_circuit(params):
    qml.BasisState(hf_state, wires=range(n_qubits_active))
    qml.AllSinglesDoubles(
        params,
        wires=range(n_qubits_active),
        hf_state=hf_state,
        singles=singles,
        doubles=doubles,
    )
    return qml.expval(H_pl)

n_params = len(singles) + len(doubles)
params = np.zeros(n_params)
opt = qml.GradientDescentOptimizer(stepsize=0.4)

energies = []
MAX_ITER = 150

for step in range(MAX_ITER):
    params, energy = opt.step_and_cost(vqe_circuit, params)
    energies.append(float(energy))
    if step % 20 == 0:
        print(f'Step {step:4d} | Energy = {energy:.8f} Ha')
    if step > 10 and abs(energies[-1] - energies[-2]) < 1e-9:
        print(f'Converged at step {step}')
        break

print(f'\nVQE  Energy : {energies[-1]:.8f} Ha')
print(f'FCI  Energy : {ethylene_data.fci_energy:.8f} Ha')
print(f'Error (mHa) : {(energies[-1] - ethylene_data.fci_energy)*1000:.4f}')

In [ ]:
# ── CELL 9: Plot convergence ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(energies, color='steelblue', linewidth=2, label='VQE Energy')
ax.axhline(ethylene_data.fci_energy, color='red', linestyle='--', label='FCI Reference')
ax.axhline(ethylene_data.hf_energy, color='gray', linestyle=':', label='HF Energy')
ax.set_xlabel('Optimizer Iteration', fontsize=12)
ax.set_ylabel('Energy (Hartree)', fontsize=12)
ax.set_title('VQE Convergence: Ethylene (STO-3G, Active Space)', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Convergence plot generated.')

In [ ]:
# ── CELL 10: Qubit count comparison across alkenes ────────────────────
molecules_info = {
    'ethylene (C2H4)':    {'n_electrons': 16, 'n_orbitals':  7},
    '1-butene (C4H8)':    {'n_electrons': 28, 'n_orbitals': 13},
    '1-hexene (C6H12)':   {'n_electrons': 40, 'n_orbitals': 19},
    '1,3-butadiene':      {'n_electrons': 28, 'n_orbitals': 13},
}

print(f"{'Molecule':<25} {'Electrons':>10} {'Orbitals':>10} {'JW Qubits':>12} {'BK Qubits':>12}")
print('-' * 70)
for mol, info in molecules_info.items():
    jw_q = info['n_orbitals'] * 2
    bk_q = jw_q  # BK ~ same, slight reduction possible
    print(f"{mol:<25} {info['n_electrons']:>10} {info['n_orbitals']:>10} {jw_q:>12} {bk_q:>12}")
print('\nNote: Active space selection can reduce qubits by 50-70% for NISQ hardware.')

## Next Steps
- **Notebook 02**: Same workflow for alkynes (acetylene, propyne)
- **Notebook 03**: Active space selection and qubit tapering to reduce qubit requirements
- **Notebook 04**: Run on IBM Quantum real hardware using Qiskit Runtime
- **Notebook 05**: Systematic benchmark comparison table (VQE vs CCSD vs FCI)